In [396]:
import pandas as pd
import numpy as np

In [397]:
df=pd.read_csv("01 covid_toy.csv")
df.head()

,age,gender,fever,cough,city,has_covid
0,60,Male,103.0,Mild,Kolkata,No
1,27,Male,100.0,Mild,Delhi,Yes
2,42,Male,101.0,Mild,Delhi,No
3,31,Female,98.0,Mild,Kolkata,No
4,65,Female,101.0,Mild,Mumbai,No


In [398]:
print(df.cough.value_counts())
print()
print()
print(df)

cough
Mild      62
Strong    38
Name: count, dtype: int64


    age  gender  fever   cough       city has_covid
0    60    Male  103.0    Mild    Kolkata        No
1    27    Male  100.0    Mild      Delhi       Yes
2    42    Male  101.0    Mild      Delhi        No
3    31  Female   98.0    Mild    Kolkata        No
4    65  Female  101.0    Mild     Mumbai        No
..  ...     ...    ...     ...        ...       ...
95   12  Female  104.0    Mild  Bangalore        No
96   51  Female  101.0  Strong    Kolkata       Yes
97   20  Female  101.0    Mild  Bangalore        No
98    5  Female   98.0  Strong     Mumbai        No
99   10  Female   98.0  Strong    Kolkata       Yes

[100 rows x 6 columns]


In [399]:
df.isnull().sum()

,0
age,0
gender,0
fever,10
cough,0
city,0
has_covid,0


In [400]:
x=df.iloc[:,0:5]
y=df.drop(x,axis='columns')
print(x.head())
print()
print()
print()
print(y.head())

   age  gender  fever cough     city
0   60    Male  103.0  Mild  Kolkata
1   27    Male  100.0  Mild    Delhi
2   42    Male  101.0  Mild    Delhi
3   31  Female   98.0  Mild  Kolkata
4   65  Female  101.0  Mild   Mumbai



  has_covid
0        No
1       Yes
2        No
3        No
4        No


In [401]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)
print(x_train.head())
print()
print()
print()

    age  gender  fever cough       city
97   20  Female  101.0  Mild  Bangalore
5    84  Female    NaN  Mild  Bangalore
40   49  Female  102.0  Mild      Delhi
95   12  Female  104.0  Mild  Bangalore
26   19  Female  100.0  Mild    Kolkata





In [402]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder

In [403]:

si = SimpleImputer(strategy='most_frequent')
le = LabelEncoder()


In [404]:
y_train['has_covid'] = le.fit_transform(y_train['has_covid'])
y_test['has_covid'] = le.transform(y_test['has_covid'])

In [405]:
x_train['fever'] = si.fit_transform(x_train[['fever']]).ravel()
x_test['fever'] = si.transform(x_test[['fever']]).ravel()

In [406]:
from sklearn.preprocessing import OrdinalEncoder

In [407]:
oe = OrdinalEncoder()

x_train[['cough']] = oe.fit_transform(x_train[['cough']])
x_test[['cough']] = oe.transform(x_test[['cough']])

In [408]:
from sklearn.preprocessing import OneHotEncoder

In [409]:
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')


In [410]:


x_train_gender_city = ohe.fit_transform(x_train[['gender', 'city']])
x_test_gender_city = ohe.transform(x_test[['gender', 'city']])

print(x_train_gender_city.shape)
print(x_test_gender_city.shape)

(80, 6)
(20, 6)


In [411]:
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder


In [412]:
transformers = [
    ('trans1', SimpleImputer(strategy='most_frequent'), ['fever']),
    ('trans2', OrdinalEncoder(categories=[['Mild', 'strong']], dtype=float), ['cough']),
    ('trans3', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), ['gender', 'city'])
]


In [413]:
ct = ColumnTransformer(transformers=transformers)

In [414]:
print(x_train['cough'].unique())
print(x_test['cough'].unique())

[0. 1.]
[0. 1.]


In [415]:

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer

# Reload the original dataset to undo previous encoding
df = pd.read_csv('01 covid_toy.csv')

x = df.drop(columns=['has_covid'])
y = df['has_covid']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42
)

# Define transformers
transformers = [
    ('trans1', SimpleImputer(strategy='median'), ['fever']),
    ('trans2', OrdinalEncoder(categories=[['Mild', 'Strong']]), ['cough']),
    ('trans3', OneHotEncoder(sparse_output=False, drop='first'), ['gender', 'city'])
]

ct = ColumnTransformer(
    transformers=transformers,
    remainder='passthrough'
)

# Fit and transform
x_train_trans = ct.fit_transform(x_train)
x_test_trans = ct.transform(x_test)

# Convert to DataFrames
columns = ct.get_feature_names_out()

x_train_trans = pd.DataFrame(x_train_trans, columns=columns)
x_test_trans = pd.DataFrame(x_test_trans, columns=columns)

# Display test data
print(x_test_trans.sample(min(5, len(x_test_trans))))
print()
print(x_test_trans.shape)


    trans1__fever  trans2__cough  trans3__gender_Male  trans3__city_Delhi  \
4           102.0            1.0                  1.0                 1.0   
14           99.0            1.0                  0.0                 1.0   
6            98.0            1.0                  0.0                 0.0   
18           99.0            1.0                  0.0                 0.0   
16          100.0            0.0                  1.0                 0.0   

    trans3__city_Kolkata  trans3__city_Mumbai  remainder__age  
4                    0.0                  0.0            20.0  
14                   0.0                  0.0            59.0  
6                    1.0                  0.0            71.0  
18                   1.0                  0.0            25.0  
16                   0.0                  0.0            80.0  

(20, 7)
